# Práctica



## Preparación previa

### Importaciones

In [ ]:
import altair as alt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

### Instalación de Altair

Para la instalación de las librerías necesarias, he creado un entorno en conda llamado `unedviz`, utilizando el comando:

```
conda create --name unedviz python=3.12
conda activate unedviz
```

Después, he procedido a instalar `altair` haciendo:
```
conda install -c conda-forge altair-all
```

Y he creado el kernel para jupyter:
```
python -m ipykernel install --user --name=unedviz --display-name "Python (unedviz)"
```

Finalmente, para comprobar la correcta instalación, he ejecutado el código de muestra de la web de Altair, instalando previamente los datasets de muestra con el comando:
```
pip install vega_datasets
```

In [ ]:
# load a sample dataset as a pandas DataFrame
from vega_datasets import data
cars = data.cars()

# make the chart
alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
).interactive()

### Carga del dataset

Los datos se encuentran en la carpeta `./CSV Files`. Estos son los archivos que tenemos:

- **`.\CSV Files\The UNSW-NB15 description.pdf`**: Archivo que describe el dataset. Indica que el conjunto de datos UNSW-NB15 fue generado en el laboratorio Cyber Range de UNSW Canberra, combinando tráfico normal real y ataques sintéticos, y contiene más de 2.5 millones de registros con 49 características, distribuidos en archivos CSV y clasificados por tipos de ataques como DoS, Exploits, y Malware. Se utilizaron herramientas como Tcpdump, Argus y Bro-IDS, y se incluyen particiones para entrenamiento (175,341 registros) y prueba (82,332 registros).
- **`.\CSV Files\NUSW-NB15_features.csv`**: Incluye una lista de 49 características, con su nombre, tipo de datos (`integer`, `nominal`, etc.) y descripción.
- **Registros de datos**: contienen los registros verdaderos de datos de tráfico real y ataques sintéticos combinados, y sus etiquetas.
    - **`.\CSV Files\UNSW-NB15_1.csv`**: Primer CSV. Contiene 49 columnas de datos, y cada columna corresponde a una de las características listadas en el archivo `NUSW-NB15_features.csv`. En total contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_2.csv`**: Segundo CSV. La estructura es igual al archivo anterior. En total contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_3.csv`**: Tercer archivo, contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_4.csv`**: Cuarto archivo, contiene 440 044 registros.
- **`.\CSV Files\NUSW-NB15_GT.csv`**: Registra una lista de los eventos de los ataques. Es el *ground truth*, contiene la verdad conocida o etiquetas reales de los datos: es decir, indica con certeza qué tipo específico de ataque es. Actúa como un archivo de referencia independiente, y con una estructura más limpia, usada para validar/relacionar los datos de otra manera. Para cada ataque, incluye su hora de inicio y hora final, la categoría de ataque (p.ej. *Backdoor*, *Exploit*, etc.), subcategoría, protocolo utilizado, IP y puerto origen, IP y puerto destino, nombre del ataque y su referencia (CVE, BID, etc.).
- **`.\CSV Files\UNSW-NB15_LIST_EVENTS.csv`**: Resumen agregado de los eventos. Contiene el número de eventos totales de cada categoría y subcategoría de ataque (datos agregados).
- **`.\CSV Files\Training and Testing Sets\UNSW_NB15_training-set.csv`**: Se trata de una partición de los archivos de datos con 175 341 registros. Sin embargo, el objetivo de este training set en concreto es utilizarlo para el entrenamiento de modelos de Machine Learning.
- **`.\CSV Files\Training and Testing Sets\UNSW_NB15_testing-set.csv`**: Igual que el training set, se trata de una partición de los archivos de datos con 82 332 registros. El objetivo de este testing set es utilizarlo para validar el entrenamiento de modelos de Machine Learning.


In [ ]:
# Cargar nombres de columnas
features_path = r'.\CSV Files\NUSW-NB15_features.csv'
features_df = pd.read_csv(features_path, encoding='latin1')
features_df.columns = features_df.columns.str.strip()

# Crear diccionario de conversión de tipos
type_mapping = {
    'Float': 'float32',
    'Integer': 'Int64',
    'integer': 'Int64',
    'Binary': 'Int8',
    'binary': 'Int8',
    'nominal': 'category',
    'Timestamp': 'str'
}

# Crear diccionario que asocie cada nombre de la columna (clave) con su tipo de datos (valor)
dtype_dict = {}
for index, row in features_df.iterrows():
    column_name = row['Name']
    column_type = type_mapping.get(row['Type'], 'object')
    dtype_dict[column_name] = column_type


In [ ]:
# Archivos de datos sin encabezado
data_files = [
    r'.\CSV Files\UNSW-NB15_1.csv',
    r'.\CSV Files\UNSW-NB15_2.csv',
    r'.\CSV Files\UNSW-NB15_3.csv',
    r'.\CSV Files\UNSW-NB15_4.csv'
]

# Leer y concatenar todos los archivos
dataframes = []
for file in data_files:
    # Leer archivo sin asignar tipos
    df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')
    dataframes.append(df)

# Concatenar los DataFrames
full_data = pd.concat(dataframes, ignore_index=True)

Carga de los datos realizado con éxito.

## Hipótesis

#### Significado de las columnas

Cada entrada del dataset representa un flujo de red (también llamado *network flow* o *network connection*). Se corresponde a una única conexión entre un dispositivo de origen y uno de destino, durante un intervalo de tiempo, con información sobre lo que ocurrió en esa conexión.

Las columnas de cada entrada del dataset UNSW-NB15 son:

| **Nombre de columna**       | **Descripción**                                                                                  |
|-----------------------------|-------------------------------------------------------------------------------------------------------------|
| `srcip`                     | Dirección IP de origen                                                                                      |
| `sport`                     | Puerto de origen                                                                                             |
| `dstip`                     | Dirección IP de destino                                                                                      |
| `dsport`                    | Puerto de destino                                                                                            |
| `proto`                     | Protocolo de la transacción                                                                                  |
| `state`                     | Estado de la conexión y protocolo asociado (p. ej., ACC, FIN, RST, etc.)                                     |
| `dur`                       | Duración total del registro                                                                                  |
| `sbytes`                    | Bytes transferidos desde el origen al destino                                                                |
| `dbytes`                    | Bytes transferidos desde el destino al origen                                                                |
| `sttl`                      | Valor TTL (time to live) de origen a destino                                                                 |
| `dttl`                      | Valor TTL de destino a origen                                                                                |
| `sloss`                     | Paquetes reenviados o perdidos desde el origen                                                               |
| `dloss`                     | Paquetes reenviados o perdidos desde el destino                                                              |
| `service`                   | Servicio usado (http, ftp, smtp, ssh, dns, etc.)                                                             |
| `Sload`                     | Bits por segundo enviados por el origen                                                                      |
| `Dload`                     | Bits por segundo recibidos en el destino                                                                     |
| `Spkts`                     | Número de paquetes enviados del origen al destino                                                            |
| `Dpkts`                     | Número de paquetes enviados del destino al origen                                                            |
| `swin`                      | Valor de ventana de anuncios TCP del origen                                                                  |
| `dwin`                      | Valor de ventana de anuncios TCP del destino                                                                 |
| `stcpb`                     | Número base de secuencia TCP del origen                                                                      |
| `dtcpb`                     | Número base de secuencia TCP del destino                                                                     |
| `smeansz`                   | Tamaño medio de los paquetes enviados por el origen                                                          |
| `dmeansz`                   | Tamaño medio de los paquetes enviados por el destino                                                         |
| `trans_depth`              | Profundidad de conexión en solicitudes/respuestas HTTP                                                       |
| `res_bdy_len`              | Tamaño del contenido real sin comprimir transferido desde el servidor HTTP                                   |
| `Sjit`                      | Jitter (variación en la latencia) del origen en milisegundos                                                 |
| `Djit`                      | Jitter del destino en milisegundos                                                                           |
| `Stime`                     | Tiempo de inicio del registro                                                                                |
| `Ltime`                     | Tiempo final del registro                                                                                     |
| `Sintpkt`                   | Tiempo entre paquetes del origen                                                                             |
| `Dintpkt`                   | Tiempo entre paquetes del destino                                                                            |
| `tcprtt`                    | Tiempo de ida y vuelta para establecer conexión TCP (synack + ackdat)                                        |
| `synack`                    | Tiempo entre SYN y SYN-ACK                                                                                   |
| `ackdat`                    | Tiempo entre SYN-ACK y ACK                                                                                   |
| `is_sm_ips_ports`          | Vale 1 si la IP y puertos de origen/destino son iguales; si no, vale 0                                       |
| `ct_state_ttl`              | Número de veces que aparece un estado particular en función de los TTL del origen y destino                  |
| `ct_flw_http_mthd`          | Número de flujos que usan métodos HTTP como GET o POST                                                       |
| `is_ftp_login`              | Vale 1 si hay acceso FTP con usuario y contraseña; si no, vale 0                                             |
| `ct_ftp_cmd`                | Número de flujos que tienen comandos FTP                                                                     |
| `ct_srv_src`                | Número de conexiones que comparten el mismo servicio y dirección IP de origen en los últimos 100 registros    |
| `ct_srv_dst`                | Número de conexiones que comparten el mismo servicio y dirección IP de destino en los últimos 100 registros   |
| `ct_dst_ltm`                | Número de conexiones con la misma dirección IP de destino en los últimos 100 registros                       |
| `ct_src_ltm`                | Número de conexiones con la misma dirección IP de origen en los últimos 100 registros                        |
| `ct_src_dport_ltm`          | Número de conexiones con la misma IP de origen y puerto de destino en los últimos 100 registros              |
| `ct_dst_sport_ltm`          | Número de conexiones con la misma IP de destino y puerto de origen en los últimos 100 registros              |
| `ct_dst_src_ltm`            | Número de conexiones entre una IP de origen y una de destino específicas en los últimos 100 registros         |
| `attack_cat`                | Categoría del ataque (Fuzzers, DoS, Shellcode, Worms, etc.)                                                  |
| `Label`                     | Etiqueta binaria: 0 = tráfico normal, 1 = tráfico malicioso                                                  |



### Amenazas y su impacto en los datos del UNSW-NB15


| **Nombre amenaza** | **Descripción** | **Alteración esperada en los datos** |
|--------------------|------------------|----------------------------------------|
| **Fuzzers** | Ataques que buscan hacer fallar la red generando paquetes con datos aleatorios. | - Incremento en `Spkts` y `Dpkts` por el envío masivo de paquetes.<br>- Aumento de `sloss` y `dloss` debido a paquetes mal formados que se pierden o descartan.<br>- `smeansz` y `dmeansz` con tamaños de paquete inusuales.<br>- Alta variabilidad en `Sjit`, `Djit`, `Sintpkt`, `Dintpkt` por comportamiento errático. |
| **Analysis** | Ataques de escaneo de puertos, spam e intentos de modificar páginas HTML. | - Elevado número de conexiones cortas con `dur` muy bajo.<br>- Posible aumento en `ct_flw_http_mthd` por múltiples métodos HTTP (GET, POST).<br>- `Sload` y `Dload` bajos pero con muchas instancias.<br>- Cambios en `state`, apareciendo muchos estados como `REQ`, `RST`, `FIN`.<br>- Incremento en `ct_srv_dst`, `ct_dst_ltm` por repetición de destino. |
| **Backdoors** | Acceso no autorizado eludiendo mecanismos de seguridad de forma sigilosa. | - `dur` largo con `Sload` y `Dload` moderados o altos (canales ocultos de datos).<br>- Actividad anormal en puertos comunes (`sport`, `dsport` inusuales).<br>- `is_sm_ips_ports` puede ser 1 (si se intenta camuflar tráfico como local).<br>- Uso de servicios poco comunes en `service`.<br>- `state` puede mostrar conexiones abiertas de largo plazo (`CON`, `INT`). |
| **DoS** | Inundación de peticiones para agotar recursos de un sistema. | - Aumento **brutal** en `Spkts`, `Dpkts`, `sbytes`, `dbytes`.<br>- `Sload` y `Dload` altísimos por congestión de red.<br>- `dur` muy bajo (peticiones rápidas, masivas).<br>- `sloss` y `dloss` altos debido a la saturación.<br>- `ct_dst_ltm`, `ct_srv_dst` elevados (muchas conexiones al mismo destino). |
| **Exploits** | Ataques que aprovechan vulnerabilidades para obtener acceso o privilegios. | - `dur` moderado-alto con `sbytes` o `dbytes` elevados en pocos registros.<br>- `stcpb`, `dtcpb` pueden mostrar patrones inusuales si se manipulan secuencias.<br>- `state` puede mostrar transiciones anómalas.<br>- `ct_ftp_cmd`, `ct_flw_http_mthd` elevados si se explotan protocolos específicos.<br>- Picos en `tcprtt`, `synack`, `ackdat` si se manipula el handshake TCP. |
| **Generic** | Ataques genéricos a cifrados de bloques, basados en conocimiento de su estructura. | - `dur` muy corto con patrones repetitivos.<br>- `smeansz`, `dmeansz` constantes o sospechosamente iguales (bloques estándar).<br>- `Sload`, `Dload` no necesariamente altos, pero frecuentes.<br>- Posible repetición de IPs o puertos (`ct_src_ltm`, `ct_dst_ltm`, etc.). |
| **Reconnaissance** | Recopilación de información (tipo escaneo) con herramientas como Strikes. | - Muchas conexiones muy cortas (`dur` muy bajo).<br>- `Spkts`, `Dpkts` bajos pero con alta frecuencia.<br>- `state` = `REQ`, `RST`, `INT` comúnmente.<br>- `ct_src_ltm`, `ct_srv_dst`, `ct_dst_ltm` muy altos por intentos a muchos destinos.<br>- Uso de puertos inusuales para escanear (`sport`, `dsport`). |
| **Shellcode** | Fragmentos de código malicioso que explotan una vulnerabilidad dentro de un programa. | - Tamaños de paquetes (`smeansz`, `dmeansz`) muy pequeños o precisos (para ejecutar payloads).<br>- `sbytes`, `dbytes` bajos en una conexión sospechosamente efectiva.<br>- `tcprtt`, `synack`, `ackdat` podrían mostrar valores alterados.<br>- `service` puede ser HTTP o FTP si se intenta inyectar vía tráfico normal.<br>- `ct_ftp_cmd` o `ct_flw_http_mthd` pueden mostrar comandos inusuales. |
| **Worms** | Malware que se auto-replica e intenta propagarse automáticamente. | - Muchísimas conexiones en poco tiempo desde la misma IP → `ct_src_ltm`, `ct_src_dport_ltm` muy altos.<br>- `Sload`, `Dload`, `Spkts`, `Dpkts` aumentados conforme se replica.<br>- `res_bdy_len` puede crecer si se transfiere el malware como archivo.<br>- Cambios en `state` mostrando múltiples inicios/terminaciones (`REQ`, `FIN`, `RST`).<br>- `is_sm_ips_ports` puede ser 1 si intenta propagarse localmente. |


## Preparación de los datos (preprocesado)

### Limpieza y asignación de tipos

In [ ]:
# Función para convertir un valor en hexadecimal a decimal y ' ' o '-' a -1
def convert_to_int(value):
    try:
        str_value = str(value).strip()
    
        if str_value.startswith('0x'): # Hexadecimal
            return int(str_value, 16)
        elif str_value in ['', '-']:
            return None
        else:
            return int(str_value)
    except ValueError:
        return value

La limpieza de los valores enteros se caracteriza sobretodo por pasar los valores hexadecimales a decimales, y por reemplazar valores como `['', '-']` con `None`.

In [ ]:
# Aplicar la conversión a enteros
for column, dtype in dtype_dict.items():
    if dtype == 'Int64':
        full_data[column] = full_data[column].apply(convert_to_int)


# Convertir las columnas al tipo correcto según el dtype_dict
for column, dtype in dtype_dict.items():
    full_data[column] = full_data[column].astype(dtype)

# Limpiar espacios en columnas tipo category
for col in full_data.select_dtypes(['category']).columns:
    full_data[col] = full_data[col].str.strip().astype('category')

In [ ]:
# Convertir las columnas Stime y Ltime a numéricas
full_data['Stime'] = pd.to_numeric(full_data['Stime'], errors='coerce')
full_data['Ltime'] = pd.to_numeric(full_data['Ltime'], errors='coerce')

# Convertir los valores de Stime y Ltime de Unix timestamp a datetime
full_data['Stime'] = pd.to_datetime(full_data['Stime'], unit='s')
full_data['Ltime'] = pd.to_datetime(full_data['Ltime'], unit='s')

In [ ]:
# Mostrar info general
print(full_data.info())

### Análisis Exploratorio de los Datos (EDA)

In [ ]:
# Valores nulos
full_data.isnull().sum()

In [ ]:
full_data.duplicated().sum()

#### Análisis de las Variables Numéricas

In [ ]:
# Ver estadística descriptiva
full_data.describe()

Ahora veremos la distribución de estas variables numéricas en formato de gráfico, para intentar obtener más detalle:

In [ ]:
alt.data_transformers.enable("vegafusion")
# Filtramos solo las columnas numéricas
numeric_cols = full_data.select_dtypes(include=['number'])

# Recorrer cada columna numérica para mostrar distribución
for col in numeric_cols.columns:
    print(f"Distribución de la columna {col}:")
        
    # Histograma
    hist = alt.Chart(full_data).mark_bar().encode(
        alt.X(f'{col}:Q',bin=alt.Bin(maxbins=300)),
        alt.Y('count():Q')
    ).properties(
        title=f'Histograma de {col}',
        width=600,
        height=400
    ).interactive()
    
    # Mostrar ambos gráficos
    hist.show()

En algunos casos parece que la distribución no tiene sentido, pero lo que realmente pasa es que tenemos outliers que dificultan la correcta visualización en el plot.

#### Análisis de las Variables Categóricas

In [ ]:
# Método para contar y calcular el % de las categorías
def count_and_percent(df, column_name):
    count_data = df[column_name].value_counts().reset_index()
    count_data.columns = [column_name, 'count']
    count_data['%'] = (count_data['count'] / count_data['count'].sum()) * 100
    return count_data

In [ ]:
# Aplicar el método a todas las columnas categóricas
categorical_columns = full_data.select_dtypes(include=['category']).columns

# Crear un diccionario o una lista para almacenar los resultados
results = {}

for col in categorical_columns:
    results[col] = count_and_percent(df, col)

# Mostrar resultados (por ejemplo, para la columna 'neighbourhood_group')
for col, result in results.items():
    print(f"Distribución para {col}:")
    print(result)
    print("\n")

## Gráficos para el Cuadro de Mando

### Resumen General

#### Total de Conexiones por Etiqueta (Normal vs Malicioso)

In [ ]:
label_count = full_data.groupby('Label').size().reset_index(name='count')
label_count['Label'] = label_count['Label'].map({0: 'Normal', 1: 'Malicious'})

label_chart = alt.Chart(label_count).mark_bar().encode(
    x=alt.X('Label:N', title='Tipo de tráfico'),
    y=alt.Y('count:Q', title='Cantidad de conexiones'),
    color='Label:N',
    tooltip=['Label:N', 'count:Q']
).properties(
    title='Distribución de tráfico: Normal vs Malicioso'
)
label_chart

#### Porcentaje de tráfico Normal vs Malicioso

In [ ]:
malicious_ratio = full_data['Label'].mean() * 100
normal_ratio = 100 - malicious_ratio

ratio_df = pd.DataFrame({
    'type': ['Normal', 'Malicious'],
    'percentage': [normal_ratio, malicious_ratio]
})

pie_chart = alt.Chart(ratio_df).mark_arc(innerRadius=50).encode(
    theta='percentage:Q',
    color='type:N',
    tooltip=['type:N', 'percentage:Q']
).properties(
    title='Porcentaje de tráfico malicioso vs normal'
)
pie_chart

#### Top 10 Categorías de Ataque

In [ ]:
attack_cat_count = full_data[full_data['Label'] == 1]['attack_cat'].value_counts().nlargest(10).reset_index()
attack_cat_count.columns = ['attack_cat', 'count']

attack_chart = alt.Chart(attack_cat_count).mark_bar().encode(
    x=alt.X('count:Q', title='Número de ataques'),
    y=alt.Y('attack_cat:N', sort='-x', title='Categoría de ataque'),
    color='attack_cat:N',
    tooltip=['attack_cat:N', 'count:Q']
).properties(
    title='Top 10 categorías de ataque'
)
attack_chart

#### Distribución de Protocolos usados

In [ ]:
proto_count = full_data['proto'].value_counts().reset_index()
proto_count.columns = ['proto', 'count']

# Separar top 5 y resto
proto_count = full_data['proto'].value_counts().reset_index()
proto_count.columns = ['proto', 'count']
top_proto = proto_count.nlargest(5, 'count')
rest_proto = proto_count[~proto_count['proto'].isin(top_proto['proto'])]

# Gráfico 1: Top 5
top_chart = alt.Chart(top_proto).mark_bar().encode(
    x=alt.X('proto:N', title='Protocolo'),
    y=alt.Y('count:Q', title='Cantidad'),
    color='proto:N',
    tooltip=['proto:N', 'count:Q']
).properties(title='Top 5 protocolos más usados')

# Gráfico 2: Resto
rest_chart = alt.Chart(rest_proto).mark_bar().encode(
    x=alt.X('proto:N', title='Protocolo'),
    y=alt.Y('count:Q', title='Cantidad'),
    color='proto:N',
    tooltip=['proto:N', 'count:Q']
).properties(title='Resto de protocolos')

top_chart & rest_chart

#### Top 10 de Servicios más utilizados

In [ ]:
service_count = full_data['service'].value_counts().nlargest(10).reset_index()
service_count.columns = ['service', 'count']

service_chart = alt.Chart(service_count).mark_bar().encode(
    x=alt.X('count:Q', title='Número de conexiones'),
    y=alt.Y('service:N', sort='-x', title='Servicio'),
    color='service:N',
    tooltip=['service:N', 'count:Q']
).properties(
    title='Top 10 servicios utilizados'
)
service_chart

### Análisis Temporal

In [ ]:
print(f"Valor mínimo de Stime: {full_data['Stime'].min()}")
print(f"Valor máximo de Stime: {full_data['Stime'].max()}")

print(f"Valor mínimo de Ltime: {full_data['Ltime'].min()}")
print(f"Valor máximo de Ltime: {full_data['Ltime'].max()}")

In [ ]:
connections_label = full_data.groupby([full_data['Stime'].dt.date, 'Label']).size().reset_index(name='count')
connections_label

#### Conexiones a lo largo del tiempo (Stime)

In [ ]:
# Agrupar por día y por etiqueta (normal o malicioso)
connections_label = full_data.groupby([full_data['Stime'].dt.date, 'Label']).size().reset_index(name='count')

# Asegurarse de que Stime es datetime
connections_label['Stime'] = pd.to_datetime(connections_label['Stime'])

# Crear rango de fechas y etiquetas posibles
all_dates = pd.date_range(start=connections_label['Stime'].min(), end=connections_label['Stime'].max())
all_labels = [0, 1]

# Producto cartesiano de fechas y etiquetas
full_index = pd.MultiIndex.from_product([all_dates, all_labels], names=['Stime', 'Label'])
full_df = pd.DataFrame(index=full_index).reset_index()

# Merge con los datos reales y rellenar faltantes con 0
combined_df_filled = pd.merge(full_df, connections_label, on=['Stime', 'Label'], how='left')
combined_df_filled['count'] = combined_df_filled['count'].fillna(0)

# Convertir Label a string para que Altair lo trate como categoría
combined_df_filled['Label'] = combined_df_filled['Label'].astype(str)

# Crear el gráfico
chart = alt.Chart(combined_df_filled).mark_line().encode(
    x=alt.X('Stime:T', title='Fecha'),
    y=alt.Y('count:Q', title='Número de Conexiones'),
    color=alt.Color('Label:N', title='Etiqueta (0: normal, 1: malicioso)'),
    tooltip=['Stime:T', 'count:Q', 'Label:N']
).properties(
    title='Conexiones por día (normales vs maliciosas)'
)

chart

Los datos sugieren que la generación de tráfico en el laboratorio fue realizada en días específicos y con un diseño progresivo. El 22 de enero se simula un entorno mayormente normal con pocos ataques, el 23 hay una caída significativa del tráfico (solo conexiones normales en bajo volumen), y el 18 de febrero se introduce una gran cantidad de tráfico malicioso y también tráfico normal. Esto indica una simulación por fases, probablemente diseñada para representar distintos escenarios: uno inicial realista, otro de baja actividad, y uno final con una oleada de ataques, posiblemente con fines de prueba o entrenamiento de sistemas de detección.

### Orígenes y Destinos

#### Top IPs de origen y destino con tráfico malicioso (`srcip` y `dstip` con `Label`=1)

In [ ]:
# IPs origen más activas (con observed=True para evitar el warning)
top_src_ips = malicious_data.groupby(['srcip', 'attack_cat'], observed=True).size().reset_index(name='count')

top_src_ips_chart = alt.Chart(top_src_ips).mark_bar().encode(
    y=alt.Y('srcip:N', sort='-x', title='IP Origen'),
    x=alt.X('count:Q', title='Número de Conexiones'),
    color=alt.Color('attack_cat:N', title='Categoría de Ataque'),
    tooltip=['srcip:N', 'attack_cat:N', 'count:Q']
).properties(
    title='IPs de Origen más Activas por Categoría de Ataque'
).transform_filter(
    alt.datum.count > 10
)

# IPs destino más activas (con observed=True también)
top_dst_ips = malicious_data.groupby(['dstip', 'attack_cat'], observed=True).size().reset_index(name='count')

top_dst_ips_chart = alt.Chart(top_dst_ips).mark_bar().encode(
    y=alt.Y('dstip:N', sort='-x', title='IP Destino'),
    x=alt.X('count:Q', title='Número de Conexiones'),
    color=alt.Color('attack_cat:N', title='Categoría de Ataque'),
    tooltip=['dstip:N', 'attack_cat:N', 'count:Q']
).properties(
    title='IPs de Destino más Activas por Categoría de Ataque'
).transform_filter(
    alt.datum.count > 10
)

top_src_ips_chart | top_dst_ips_chart

### Análisis de Flujos

#### Distribución de duración de las conexiones (`dur`)

In [ ]:
dur_chart = alt.Chart(full_data).mark_bar().encode(
    x=alt.X('dur:Q', bin=True, title='Duración de Conexiones (segundos)'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    tooltip=['dur:Q', 'count():Q']
).properties(
    title='Distribución de Duración de las Conexiones'
)
dur_chart

In [ ]:
# Conexiones cortas (≤ 60 segundos)
short_dur = full_data[full_data['dur'] <= 60]

short_chart = alt.Chart(short_dur).mark_bar().encode(
    x=alt.X('dur:Q', bin=alt.Bin(maxbins=50), title='Duración ≤ 60s'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    tooltip=['dur:Q', 'count():Q']
).properties(title='Distribución de Conexiones Cortas (≤ 60s)')

# Conexiones largas (> 60 segundos)
long_dur = full_data[full_data['dur'] > 60]

long_chart = alt.Chart(long_dur).mark_bar().encode(
    x=alt.X('dur:Q', bin=alt.Bin(maxbins=50), title='Duración > 60s'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    tooltip=['dur:Q', 'count():Q']
).properties(title='Distribución de Conexiones Largas (> 60s)')

# Mostrar ambos gráficos uno al lado del otro
short_chart | long_chart

In [ ]:
# Rellenar valores nulos en 'attack_cat' con 'NA' para identificar conexiones normales
if full_data['attack_cat'].dtype.name == 'category':
    full_data['attack_cat'] = full_data['attack_cat'].cat.add_categories('NA')

In [ ]:
full_data['attack_cat'] = full_data['attack_cat'].fillna('NA')

# Conexiones cortas (≤ 60 segundos)
short_dur = full_data[full_data['dur'] <= 60]

short_chart = alt.Chart(short_dur).mark_bar().encode(
    x=alt.X('dur:Q', bin=alt.Bin(maxbins=50), title='Duración ≤ 60s'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    color=alt.Color('attack_cat:N', title='Categoría de Ataque'),
    tooltip=['dur:Q', 'count():Q', 'attack_cat:N']
).properties(title='Distribución de Conexiones Cortas (≤ 60s)')

# Conexiones largas (> 60 segundos)
long_dur = full_data[full_data['dur'] > 60]

long_chart = alt.Chart(long_dur).mark_bar().encode(
    x=alt.X('dur:Q', bin=alt.Bin(maxbins=50), title='Duración > 60s'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    color=alt.Color('attack_cat:N', title='Categoría de Ataque'),
    tooltip=['dur:Q', 'count():Q', 'attack_cat:N']
).properties(title='Distribución de Conexiones Largas (> 60s)')

# Mostrar ambos gráficos uno al lado del otro
short_chart | long_chart

#### Transferencia de bytes origen/destino (`sbytes`, `dbytes`)

In [ ]:
# Agrupar y sumar sbytes y dbytes por día
transfer_bytes = full_data.groupby(full_data['Stime'].dt.date)[['sbytes', 'dbytes']].sum().reset_index()
transfer_bytes['Stime'] = pd.to_datetime(transfer_bytes['Stime'])

# Crear rango de fechas completo
date_range = pd.date_range(start=transfer_bytes['Stime'].min(), end=transfer_bytes['Stime'].max())

# Crear DataFrame base con todas las fechas
base = pd.DataFrame({'Stime': date_range})

# Merge con los datos y rellenar con 0 los días sin tráfico
transfer_bytes_filled = base.merge(transfer_bytes, on='Stime', how='left').fillna(0)

# Reorganizar para Altair (tipo largo)
transfer_bytes_long = transfer_bytes_filled.melt(id_vars='Stime', value_vars=['sbytes', 'dbytes'],
                                                 var_name='Tipo', value_name='Bytes')

# Crear el gráfico
bytes_chart = alt.Chart(transfer_bytes_long).mark_line().encode(
    x=alt.X('Stime:T', title='Fecha'),
    y=alt.Y('Bytes:Q', title='Bytes Transferidos'),
    color=alt.Color('Tipo:N', title='Tipo de Tráfico', scale=alt.Scale(scheme='tableau10')),
    tooltip=['Stime:T', 'Tipo:N', 'Bytes:Q']
).properties(
    title='Transferencia de Bytes por Día (Origen y Destino)'
)

bytes_chart

#### Velocidad de transferencia (`Sload`, `Dload`)

In [ ]:
# Asegurar formato datetime
full_data['Stime'] = pd.to_datetime(full_data['Stime'])

# Agrupar por fecha y categoría de ataque, calcular la media (con observed=True)
avg_speed = full_data.groupby(
    [full_data['Stime'].dt.date, 'attack_cat'], observed=True
)[['Sload', 'Dload']].mean().reset_index()
avg_speed['Stime'] = pd.to_datetime(avg_speed['Stime'])

# Gráfico para Sload (velocidad media desde origen)
sload_chart = alt.Chart(avg_speed).mark_line().encode(
    x=alt.X('Stime:T', title='Fecha'),
    y=alt.Y('Sload:Q', title='Velocidad Media desde Origen (bits/s)'),
    color=alt.Color('attack_cat:N', legend=alt.Legend(title="Categoría de Ataque")),
    tooltip=['Stime:T', 'attack_cat:N', 'Sload:Q']
).properties(
    title='Velocidad Media desde Origen por Categoría de Ataque'
)

# Gráfico para Dload (velocidad media hacia destino)
dload_chart = alt.Chart(avg_speed).mark_line().encode(
    x=alt.X('Stime:T', title='Fecha'),
    y=alt.Y('Dload:Q', title='Velocidad Media hacia Destino (bits/s)'),
    color=alt.Color('attack_cat:N', legend=alt.Legend(title="Categoría de Ataque")),
    tooltip=['Stime:T', 'attack_cat:N', 'Dload:Q']
).properties(
    title='Velocidad Media hacia Destino por Categoría de Ataque'
)

# Mostrar ambos gráficos juntos
sload_chart | dload_chart

### Análisis por Puertos y Protocolo

#### Puertos más usados (origen y destino) (`sport`, `dsport`)

In [ ]:
top_sport = full_data['sport'].value_counts().reset_index(name='count').head(10)
top_sport.columns = ['sport', 'count']

# Gráfico para Puertos más usados (origen)
sport_chart = alt.Chart(top_sport).mark_bar().encode(
    x=alt.X('count:Q', title='Número de Conexiones'),
    y=alt.Y('sport:N', title='Puerto de Origen', sort='-x'),
    tooltip=['sport:N', 'count:Q']
).properties(
    title='Top 10 Puertos de Origen más Usados'
)
sport_chart

In [ ]:
top_dsport = full_data['dsport'].value_counts().reset_index(name='count').head(10)
top_dsport.columns = ['dsport', 'count']

# Gráfico para Puertos más usados (destino)
dsport_chart = alt.Chart(top_dsport).mark_bar().encode(
    x=alt.X('count:Q', title='Número de Conexiones'),
    y=alt.Y('dsport:N', title='Puerto de Destino', sort='-x'),
    tooltip=['dsport:N', 'count:Q']
).properties(
    title='Top 10 Puertos de Destino más Usados'
)
dsport_chart

#### Combinaciones comunes de puertos/protocolos en ataques

In [ ]:
# Filtrar tráfico malicioso
malicious_data = full_data[full_data['Label'] == 1]

# Limitar los datos a puertos relevantes (solo los más comunes)
top_sports = malicious_data['sport'].value_counts().head(100).index
top_dsports = malicious_data['dsport'].value_counts().head(100).index

# Filtrar los datos solo para estos puertos
filtered_data = malicious_data[malicious_data['sport'].isin(top_sports) & malicious_data['dsport'].isin(top_dsports)].copy()

# Crear una nueva columna combinada para las combinaciones de puertos y protocolos
filtered_data['sport_dsport_proto'] = filtered_data['sport'].astype(str) + '-' + filtered_data['dsport'].astype(str) + '-' + filtered_data['proto'].astype(str)

# Agrupar por la nueva columna combinada y contar las ocurrencias
top_ports_protocols = filtered_data.groupby('sport_dsport_proto').size().reset_index(name='count')

# Ordenar y seleccionar las 10 combinaciones más comunes
top_ports_protocols = top_ports_protocols.sort_values(by='count', ascending=False).head(10)

# Crear gráfico para las combinaciones más comunes
ports_protocols_chart = alt.Chart(top_ports_protocols).mark_bar().encode(
    x=alt.X('count:Q', title='Número de Conexiones'),
    y=alt.Y('sport_dsport_proto:N', title='Combinación de Puerto Origen - Destino - Protocolo', sort='-x'),
    color=alt.Color('sport_dsport_proto:N', title='Combinación (Origen-Destino-Protocolo)'),
    tooltip=['sport_dsport_proto:N', 'count:Q']
).properties(
    title='Combinaciones Comunes de Puertos/Protocolos en Ataques'
)

ports_protocols_chart

### Detección de Patrones Raros

#### Conexiones con `is_sm_ips_ports = 1`

El campo is_sm_ips_ports es un indicador booleano que vale 1 cuando lla IP y los puertos de origen y destino son exactamente iguales (`src_ip == dst_ip` y `sport == dsport`). Esto no necesariamente implica actividad maliciosa, pero es una rareza en tráfico de red, ya que en redes normales, es poco común que una conexión vaya de un puerto a sí mismo con la misma IP. Por tanto, puede señalar patrones inusuales o anómalos que vale la pena investigar.

Es interesante contar cuántas hay por categoría de ataque:

In [ ]:
# Filtrar conexiones sospechosas
suspicious_connections = full_data[full_data['is_sm_ips_ports'] == 1].copy()

# Rellenar valores nulos en 'attack_cat' si los hubiera
suspicious_connections['attack_cat'] = suspicious_connections['attack_cat'].fillna('NA')

# Gráfico de barras por tipo de ataque
suspicious_chart = alt.Chart(suspicious_connections).mark_bar().encode(
    x=alt.X('count():Q', title='Número de Conexiones Sospechosas'),
    y=alt.Y('attack_cat:N', sort='-x', title='Categoría de Ataque'),
    color=alt.Color('attack_cat:N', title='Categoría de Ataque'),
    tooltip=['attack_cat:N', 'count():Q']
).properties(
    title='Conexiones con is_sm_ips_ports = 1'
)

suspicious_chart

El hecho de que ninguna conexión con `is_sm_ips_ports = 1` esté etiquetada como ataque sugiere que esta condición no es un buen indicador de actividad maliciosa en este conjunto de datos. Podría tratarse de tráfico legítimo generado por pruebas, servicios internos o comportamiento normal del sistema.

#### Conexiones con jitter elevado (`Sjit`, `Djit`)

Los histogramas de jitter permiten visualizar la variabilidad en los tiempos de transmisión desde el origen o destino, identificando si hay retrasos inusuales o fluctuaciones. Esto ayuda a detectar posibles problemas de red o comportamientos anómalos asociados a ciertos tipos de tráfico.

El histograma que cuenta cuántas conexiones tienen un determinado rango de jitter desde el origen (`Sjit`) o hacia el destino (`Djit`). El objetivo de estos gráficos es visualizar si el jitter está concentrado en valores bajos (lo normal) o si hay muchas conexiones con jitter alto, lo cual podría ser señal de congestión, mala calidad de red o incluso actividades maliciosas.

In [ ]:
# Opcional: limitar valores extremos para mejor visualización
filtered_jitter = full_data[(full_data['Sjit'] < 1e6) & (full_data['Djit'] < 1e6)].copy()

# Histograma de Sjit
sjit_chart = alt.Chart(filtered_jitter).mark_bar().encode(
    x=alt.X('Sjit:Q', bin=alt.Bin(maxbins=50), title='Jitter desde Origen (Sjit)'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    tooltip=['count():Q']
).properties(title='Distribución de Sjit')

# Histograma de Djit
djit_chart = alt.Chart(filtered_jitter).mark_bar().encode(
    x=alt.X('Djit:Q', bin=alt.Bin(maxbins=50), title='Jitter hacia Destino (Djit)'),
    y=alt.Y('count():Q', title='Número de Conexiones'),
    tooltip=['count():Q']
).properties(title='Distribución de Djit')

sjit_chart | djit_chart

#### Análisis de tiempos TCP anómalos (`tcprtt`, `synack`, `ackdat`)

El objetivo de este gráfico de dispersión es analizar la relación entre el **tiempo total de ida y vuelta TCP (`tcprtt`)** y el **tiempo que tarda en recibirse un SYN-ACK (`synack`)** para diferentes categorías de tráfico (incluyendo ataques y tráfico normal). Al representar cada conexión como un punto y colorearlo según su `attack_cat`, el gráfico permite detectar **comportamientos anómalos o patrones específicos** de ciertas categorías de ataque, como tiempos de respuesta anormalmente altos o agrupaciones distintivas. Filtrar los valores extremos mejora la visualización, evitando que unos pocos datos extremos distorsionen el análisis visual.

In [ ]:
# Filtrar valores extremos para visualizar mejor
tcp_times = full_data[(full_data['tcprtt'] < 5e6) & 
                      (full_data['synack'] < 5e6) & 
                      (full_data['ackdat'] < 5e6)].copy()

# Convertir attack_cat para evitar nulos
tcp_times['attack_cat'] = tcp_times['attack_cat'].fillna('NA')

# Gráfico de dispersión tcprtt vs synack
rtt_scatter = alt.Chart(tcp_times).mark_circle(size=60, opacity=0.5).encode(
    x=alt.X('tcprtt:Q', title='TCP Round Trip Time'),
    y=alt.Y('synack:Q', title='Tiempo SYN-ACK'),
    color=alt.Color('attack_cat:N', title='Categoría de Ataque'),
    tooltip=['attack_cat:N', 'tcprtt:Q', 'synack:Q', 'ackdat:Q']
).properties(
    title='Relación entre TCP RTT y SYN-ACK'
)

rtt_scatter

Objetivo de un Cuadro de Mandos

- **Visión general del tráfico de red** (normal vs malicioso).
- **Detección de anomalías**.
- **Análisis por tipo de ataque**.
- **Análisis por IP origen/destino o servicio/protocolo**.
- **Estadísticas temporales**.

